# k-means clustering

How to run it, how to get a better answer out of it, how to choose k,
and when to stop trusting it.

## 1. Running k-means

1797 handwritten digits, each an 8x8 image of 64 numbers. There are ten
kinds, and k-means is never told that or shown a single label.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits

digits = load_digits()
X = digits.data
print(X.shape)

Look at the data first. Each row of 64 numbers is an image, so it can be
drawn. Here is one of each kind:

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for d, ax in enumerate(axes):
    ax.imshow(X[digits.target == d][0].reshape(8, 8), cmap='gray_r')
    ax.set_title(d, fontsize=9)
    ax.axis('off')
plt.show()

`digits.target` was used only to choose which images to show. k-means never
sees it.

64 columns is too many to plot. PCA from the last unit takes it down to two,
which is enough to look at.

In [ ]:
from sklearn.decomposition import PCA

flat = PCA(n_components=2).fit_transform(X)

plt.figure(figsize=(5, 4.5))
plt.scatter(flat[:, 0], flat[:, 1], s=8, c='black')
plt.title('the digits in 2 components')
plt.show()

Some structure, no labels on it. Now cluster.

In [ ]:
km = KMeans(n_clusters=10, random_state=6).fit(X)
print('cost:', round(km.inertia_))

Draw the same picture again, colored by the group k-means put each point in.

In [ ]:
def show_clusters(points, labels, title):
    plt.figure(figsize=(5, 4.5))
    plt.scatter(points[:, 0], points[:, 1], s=8, c=labels, cmap='tab10')
    plt.title(title)
    plt.show()

show_clusters(flat, km.labels_, 'colored by k-means group')

Each center is 64 numbers, so it can be drawn as an 8x8 image: the average
digit of its group.

In [ ]:
def show_centers(km, title):
    fig, axes = plt.subplots(1, km.n_clusters, figsize=(km.n_clusters, 1.4))
    for ax, c in zip(axes, km.cluster_centers_):
        ax.imshow(c.reshape(8, 8), cmap='gray_r')
        ax.axis('off')
    fig.suptitle(title)
    plt.show()

show_centers(km, 'cost = %d' % round(km.inertia_))

Some of those are clean digits. Some are two digits printed on top of each
other, which means one group is holding more than one kind of digit.

### Your turn

`KMeans` takes these arguments:

`n_init`  ·  `init`  ·  `max_iter`  ·  `tol`  ·  `algorithm`  ·  `random_state`

Look them up: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html

Then change them and get the cost below 1,218,629, keeping `n_clusters=10`.
**How low can you get it, and which argument did the work?**

Redraw the two pictures when you have a better answer and see what moved.

In [ ]:
km = KMeans(n_clusters=10, random_state=6).fit(X)
print(round(km.inertia_))


Two questions to answer from your own results, not from the documentation:

1. Several of those arguments changed nothing at all. Which ones, and why?
2. The one that worked — what is it doing, and what does that say about the
   bad start you ran by hand on the worksheet?

## 2. Choosing k

With the digits we already knew the answer was ten. Most of the time nobody
tells you.

The fix is the one from section 1: you cannot tell from a single run, so run
it many times and compare the cost. There it was many starting points at one
k. Here it is a range of k. Pick a sensible range, loop over it, keep every
cost, and look at the curve.

In [ ]:
from sklearn.datasets import make_blobs

data, _ = make_blobs(n_samples=400, centers=3, cluster_std=1.0,
                     random_state=42)

plt.figure(figsize=(5, 4))
plt.scatter(data[:, 0], data[:, 1], s=12, c='black')
plt.title('how many groups?')
plt.show()

In [ ]:
ks = range(1, 9)
cost = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(data)
    cost.append(km.inertia_)

plt.figure(figsize=(5, 3.5))
plt.plot(ks, cost, 'o-')
plt.xlabel('k')
plt.ylabel('cost')
plt.show()

The cost drops steeply up to a point and then flattens out. That corner is
the elbow, and it is where adding another group stops buying you much.
Here it is at k = 3, which is what the picture shows.

**Why not just take the k with the smallest cost?**

In [ ]:
km = KMeans(n_clusters=len(data), n_init=1, random_state=0).fit(data)
print('k =', len(data), ' cost =', round(km.inertia_, 6))

Zero. Give every point its own group and nothing is ever far from its center.
The cost will always reward more groups, so the smallest cost is always the
largest k, and that is never the answer. **The range you loop over has to be
sensible to begin with** — the curve can only pick between the options you
give it.

## 3. Where k-means fails

Everything above assumed k-means was the right tool for the data. Two cases
where it is not.

### Shapes that are not blobs

In [ ]:
from sklearn.datasets import make_circles

rings, truth = make_circles(n_samples=400, factor=0.4, noise=0.06,
                            random_state=0)
labels = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(rings)

fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
ax[0].scatter(rings[:, 0], rings[:, 1], s=10, c=truth, cmap='coolwarm')
ax[0].set_title('the two groups')
ax[1].scatter(rings[:, 0], rings[:, 1], s=10, c=labels, cmap='coolwarm')
ax[1].set_title('what k-means found')
for a in ax:
    a.set_aspect('equal')
plt.show()

It cut the picture in half. Every boundary k-means can draw is a straight
line halfway between two centers, and no straight line separates a ring from
the ring around it. This is the same shape that defeated a single straight
boundary back in Unit 61.

### Data with no groups in it

Points scattered at random, with no structure of any kind.

In [ ]:
rng = np.random.default_rng(0)
noise = rng.uniform(0, 10, size=(400, 2))
labels = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(noise)

fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
ax[0].scatter(noise[:, 0], noise[:, 1], s=10, c='black')
ax[0].set_title('the data')
ax[1].scatter(noise[:, 0], noise[:, 1], s=10, c=labels, cmap='tab10')
ax[1].set_title('k-means, k = 4')
plt.show()

Four tidy groups, cleanly drawn, and not one of them means anything.
k-means always returns exactly the k you asked for. Nothing in the output
says *there was nothing here*.

Worse, the elbow does not save you either:

In [ ]:
ks = range(1, 9)
cost = [KMeans(n_clusters=k, n_init=10, random_state=0).fit(noise).inertia_
        for k in ks]

plt.figure(figsize=(5, 3.5))
plt.plot(ks, cost, 'o-')
plt.xlabel('k')
plt.ylabel('cost')
plt.title('random noise, no groups')
plt.show()

**That curve bends too.** Read on its own it would have you believing in
groups that do not exist. The elbow tells you where the cost stops falling,
not whether there was anything there to find.

## Your turn

The wine data. `wine.target` holds the variety of each bottle — do not give
it to k-means, and use it only at the end.

1. Is k-means the right tool here? How can you tell?
2. What value of k do you suggest? And how do you find it?
3. Plot the two PCA components again, this time colored by the groups
   k-means found.
4. Mark it properly. Compare your groups with `wine.target` using
   `confusion_matrix`. Report your observation and your assessment.

*Make sure you scale the measurements first.*

In [ ]:
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix

wine = load_wine()
